# Bài 5: Mô hình hỗn hợp Gaussian (GMM) với ước lượng tham số EM

## 1. Mô tả bài toán
- Cài đặt thuật toán **Kỳ vọng - Tối đa hóa (Expectation-Maximization - EM)** từ đầu bằng **NumPy** để ước lượng các tham số của **Mô hình hỗn hợp Gaussian (Gaussian Mixture Model - GMM)** cho bài toán phân cụm không giám sát (Unsupervised Clustering).
- Quan sát sự hội tụ và tiến hóa của các cụm thông qua **đường đồng mức (contour plots)** trên tập dữ liệu 2D giả lập qua từng vòng lặp EM.

## 2. Cơ sở lý thuyết & Công thức toán học

### a. Mô hình hỗn hợp Gaussian (GMM):
Giả sử tập dữ liệu $X = \{x_1, x_2, \dots, x_N\} \subset \mathbb{R}^D$ được sinh ra từ hỗn hợp của $K$ phân phối chuẩn nhiều chiều:
$$p(x) = \sum_{k=1}^K \pi_k \mathcal{N}(x \mid \mu_k, \Sigma_k)$$
Trong đó:
- $\pi_k = P(z = k)$ là trọng số cụm (mixing coefficients), thỏa mãn $\pi_k \ge 0$ và $\sum_{k=1}^K \pi_k = 1$.
- $\mu_k \in \mathbb{R}^D$ là vector kỳ vọng (tâm cụm) của thành phần thứ $k$.
- $\Sigma_k \in \mathbb{R}^{D \times D}$ là ma trận hiệp phương sai (hình dạng và hướng elip của cụm), đối xứng và xác định dương.
- Hàm mật độ phân phối chuẩn $D$ chiều:
  $$\mathcal{N}(x \mid \mu_k, \Sigma_k) = \frac{1}{(2\pi)^{D/2} |\Sigma_k|^{1/2}} \exp\left( -\frac{1}{2}(x - \mu_k)^T \Sigma_k^{-1} (x - \mu_k) \right)$$

### b. Hàm Log-Likelihood chưa hoàn chỉnh (Incomplete Log-Likelihood):
$$\ln p(X \mid \pi, \mu, \Sigma) = \sum_{i=1}^N \ln \left( \sum_{k=1}^K \pi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right)$$
Vì có tổng nằm bên trong hàm logarit, ta không thể tìm nghiệm giải tích trực tiếp. Do đó, thuật toán **EM (Expectation-Maximization)** được sử dụng.

### c. Bước E (Expectation step) - Yêu cầu 1:
Tính xác suất hậu nghiệm (hay trách nhiệm - *responsibilities*) $\gamma_{ik} = P(z_i = k \mid x_i)$ thể hiện xác suất điểm $x_i$ thuộc về thành phần Gaussian thứ $k$:
$$\gamma_{ik} = \frac{\pi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \pi_j \mathcal{N}(x_i \mid \mu_j, \Sigma_j)}, \quad \text{với } \sum_{k=1}^K \gamma_{ik} = 1$$
Tổng trách nhiệm của cụm $k$ (số điểm hiệu dụng thuộc cụm $k$):
$$N_k = \sum_{i=1}^N \gamma_{ik}$$

### d. Bước M (Maximization step - Weighted MLE) - Yêu cầu 2:
Cập nhật lại các tham số của $K$ phân phối chuẩn dựa trên trọng số xác suất $\gamma_{ik}$:
1. **Cập nhật trọng số cụm:**
   $$\pi_k^{new} = \frac{N_k}{N} = \frac{1}{N} \sum_{i=1}^N \gamma_{ik}$$
2. **Cập nhật vector kỳ vọng (tâm cụm):**
   $$\mu_k^{new} = \frac{1}{N_k} \sum_{i=1}^N \gamma_{ik} x_i$$
3. **Cập nhật ma trận hiệp phương sai:**
   $$\Sigma_k^{new} = \frac{1}{N_k} \sum_{i=1}^N \gamma_{ik} (x_i - \mu_k^{new})(x_i - \mu_k^{new})^T$$

In [ ]:
# 0. Import các thư viện cần thiết
import numpy as np
import matplotlib.pyplot as plt

# Cố định random seed để kết quả mô phỏng có thể tái lập
np.random.seed(42)

In [ ]:
# ==============================================================================
# YÊU CẦU 1 & 2: Cài đặt hàm mật độ Gaussian và Lớp GMM với thuật toán EM
# ==============================================================================

def multivariate_gaussian_pdf(X, mean, cov):
    """
    Tính hàm mật độ xác suất phân phối chuẩn nhiều chiều N(x | mean, cov).
    """
    d = len(mean)
    # Thêm epsilon vào đường chéo để ổn định số học chống suy biến ma trận hiệp phương sai
    cov_reg = cov + 1e-6 * np.eye(d)
    diff = X - mean
    inv_cov = np.linalg.inv(cov_reg)
    det_cov = np.linalg.det(cov_reg)
    
    norm_const = 1.0 / (np.power(2.0 * np.pi, d / 2.0) * np.sqrt(np.maximum(det_cov, 1e-12)))
    exponent = -0.5 * np.sum(diff @ inv_cov * diff, axis=1)
    return norm_const * np.exp(np.clip(exponent, -50.0, 50.0))

class GaussianMixtureModel:
    """
    Mô hình hỗn hợp Gaussian (GMM) ước lượng bằng thuật toán EM (Expectation-Maximization).
    """
    def __init__(self, K=3, max_iters=50, tol=1e-4):
        self.K = K
        self.max_iters = max_iters
        self.tol = tol
        self.pi = None       # Trọng số cụm pi_k
        self.mu = None       # Vector kỳ vọng mu_k
        self.sigma = None    # Ma trận hiệp phương sai Sigma_k
        self.gamma = None    # Xác suất hậu nghiệm gamma_ik
        self.log_likelihood_history = []
        self.snapshots = []  # Lưu lại trạng thái qua các vòng lặp để vẽ đường đồng mức

    def fit(self, X):
        N, D = X.shape
        
        # --- KHỞI TẠO THAM SỐ (Initialization) ---
        # Chọn ngẫu nhiên K điểm dữ liệu làm tâm kỳ vọng ban đầu
        rand_indices = np.random.choice(N, self.K, replace=False)
        self.mu = X[rand_indices].copy()
        
        # Ma trận hiệp phương sai khởi tạo theo phương sai toàn cục của dữ liệu
        global_var = np.var(X, axis=0)
        self.sigma = np.array([np.diag(global_var) for _ in range(self.K)])
        
        # Trọng số tiên nghiệm khởi tạo đều nhau
        self.pi = np.ones(self.K) / self.K

        for it in range(self.max_iters):
            # ==================================================================
            # YÊU CẦU 1: BƯỚC E (Expectation Step)
            # Tính xác suất hậu nghiệm gamma_ik = P(z_i = k | x_i)
            # ==================================================================
            weighted_pdf = np.zeros((N, self.K))
            for k in range(self.K):
                weighted_pdf[:, k] = self.pi[k] * multivariate_gaussian_pdf(X, self.mu[k], self.sigma[k])
            
            total_density = np.sum(weighted_pdf, axis=1, keepdims=True)
            total_density = np.maximum(total_density, 1e-12)  # Tránh chia cho 0
            
            self.gamma = weighted_pdf / total_density  # shape (N, K)

            # Tính Log-Likelihood của toàn bộ dữ liệu
            log_likelihood = np.sum(np.log(total_density))
            self.log_likelihood_history.append(log_likelihood)

            # Lưu lại snapshot tại các vòng lặp quan trọng để trực quan hóa
            if it in [0, 1, 2, 4, 9, 14] or it == self.max_iters - 1:
                self.snapshots.append({
                    'iter': it + 1,
                    'mu': self.mu.copy(),
                    'sigma': self.sigma.copy(),
                    'pi': self.pi.copy(),
                    'gamma': self.gamma.copy(),
                    'll': log_likelihood
                })

            # Kiểm tra tiêu chuẩn hội tụ
            if it > 0 and abs(self.log_likelihood_history[-1] - self.log_likelihood_history[-2]) < self.tol:
                print(f"Thuật toán EM đã hội tụ tại vòng lặp thứ {it + 1}!")
                break

            # ==================================================================
            # YÊU CẦU 2: BƯỚC M (Maximization Step - Weighted MLE)
            # Cập nhật pi_k, mu_k, Sigma_k
            # ==================================================================
            N_k = np.sum(self.gamma, axis=0)  # Tổng trách nhiệm của từng cụm (shape: K)

            # 1. Cập nhật trọng số cụm: pi_k = N_k / N
            self.pi = N_k / N

            # 2. Cập nhật vector kỳ vọng: mu_k = (1 / N_k) * sum_i (gamma_ik * x_i)
            for k in range(self.K):
                self.mu[k] = np.sum(self.gamma[:, k:k+1] * X, axis=0) / N_k[k]

            # 3. Cập nhật ma trận hiệp phương sai:
            #    Sigma_k = (1 / N_k) * sum_i gamma_ik * (x_i - mu_k)(x_i - mu_k)^T
            for k in range(self.K):
                diff = X - self.mu[k]
                weighted_diff = self.gamma[:, k:k+1] * diff
                self.sigma[k] = (weighted_diff.T @ diff) / N_k[k]
                # Regularization cộng epsilon vào đường chéo chống suy biến
                self.sigma[k] += 1e-6 * np.eye(D)

        return self

In [ ]:
# ==============================================================================
# YÊU CẦU 3: Tạo tập dữ liệu 2D giả lập gồm 3 cụm Gaussian có hình elip xoay
# ==============================================================================
n_per_cluster = 150

# Cụm 1: Hướng nghiêng chéo dương
true_mu1 = np.array([1.0, 2.0])
true_cov1 = np.array([[1.5, 0.9], [0.9, 1.0]])

# Cụm 2: Dạng elip dẹt nằm ngang phía dưới
true_mu2 = np.array([7.0, 1.5])
true_cov2 = np.array([[2.0, -0.6], [-0.6, 0.5]])

# Cụm 3: Cụm phía trên cao
true_mu3 = np.array([4.0, 7.0])
true_cov3 = np.array([[0.8, 0.2], [0.2, 1.2]])

X1 = np.random.multivariate_normal(true_mu1, true_cov1, n_per_cluster)
X2 = np.random.multivariate_normal(true_mu2, true_cov2, n_per_cluster)
X3 = np.random.multivariate_normal(true_mu3, true_cov3, n_per_cluster)
X = np.vstack([X1, X2, X3])

print(f"Tổng số mẫu dữ liệu: {len(X)} điểm (mỗi cụm {n_per_cluster} điểm).")
print(f"Tọa độ tâm thực tế (Ground Truth Means):")
print(f"  Cụm 1: {true_mu1}")
print(f"  Cụm 2: {true_mu2}")
print(f"  Cụm 3: {true_mu3}")

In [ ]:
# ==============================================================================
# YÊU CẦU 3: Áp dụng thuật toán EM và Đánh giá tham số ước lượng
# ==============================================================================
gmm = GaussianMixtureModel(K=3, max_iters=50, tol=1e-4)
gmm.fit(X)

print("\n" + "=" * 75)
print("KẾT QUẢ ƯỚC LƯỢNG THAM SỐ GMM SAU KHI HỘI TỤ:")
print("=" * 75)
for k in range(gmm.K):
    print(f"Cụm {k+1}:")
    print(f"  - Trọng số pi_{k+1}:  {gmm.pi[k]:.4f} (Thực tế: 0.3333)")
    print(f"  - Tâm kỳ vọng mu_{k+1}: [{gmm.mu[k, 0]:.3f}, {gmm.mu[k, 1]:.3f}]")
    print(f"  - Ma trận hiệp phương sai Sigma_{k+1}:")
    print(f"    [{gmm.sigma[k, 0, 0]:.3f}, {gmm.sigma[k, 0, 1]:.3f}]")
    print(f"    [{gmm.sigma[k, 1, 0]:.3f}, {gmm.sigma[k, 1, 1]:.3f}]")
print("=" * 75)

In [ ]:
# ==============================================================================
# YÊU CẦU 3: Trực quan hóa các đường đồng mức (Contour Plots) qua các vòng lặp EM
# ==============================================================================
selected_snapshots = gmm.snapshots[:4]  # Chọn 4 mốc vòng lặp tiêu biểu
n_plots = len(selected_snapshots)

fig, axes = plt.subplots(1, n_plots, figsize=(5.5 * n_plots, 5), dpi=120)

# Thiết lập lưới tọa độ để vẽ đường đồng mức
x_min, x_max = X[:, 0].min() - 1.0, X[:, 0].max() + 1.0
y_min, y_max = X[:, 1].min() - 1.0, X[:, 1].max() + 1.0
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 120), np.linspace(y_min, y_max, 120))
grid_points = np.column_stack([xx.ravel(), yy.ravel()])

colors = ['#e74c3c', '#2ecc71', '#3498db']  # Màu cho 3 cụm

for idx, snap in enumerate(selected_snapshots):
    ax = axes[idx]
    
    # Gán nhãn cụm mềm dựa trên trách nhiệm gamma lớn nhất
    hard_labels = np.argmax(snap['gamma'], axis=1)
    
    # Vẽ các điểm dữ liệu theo màu cụm
    for k in range(gmm.K):
        cluster_points = X[hard_labels == k]
        ax.scatter(cluster_points[:, 0], cluster_points[:, 1], 
                   s=20, color=colors[k], alpha=0.55, edgecolors='none')
        
        # Tính hàm mật độ trên lưới để vẽ đường đồng mức (Contour)
        pdf_grid = multivariate_gaussian_pdf(grid_points, snap['mu'][k], snap['sigma'][k])
        Z = pdf_grid.reshape(xx.shape)
        
        # Vẽ 3 mức đường đồng mức (đại diện cho các khoảng xác suất)
        ax.contour(xx, yy, Z, levels=3, colors=colors[k], linewidths=2.2, alpha=0.9)
        
        # Đánh dấu tâm cụm mu_k
        ax.scatter(snap['mu'][k, 0], snap['mu'][k, 1], 
                   marker='X', s=120, color='black', edgecolors='white', linewidth=1.5, zorder=5)
    
    ax.set_title(f"Vòng lặp EM: {snap['iter']}\nLog-Likelihood: {snap['ll']:.1f}", 
                 fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('$x_1$', fontsize=11)
    ax.set_ylabel('$x_2$', fontsize=11)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.grid(True, linestyle=':', alpha=0.5)

plt.suptitle('Sự tiến hóa của các đường đồng mức Gaussian qua các vòng lặp EM', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ĐỒ THỊ BỔ SUNG: Đường cong hội tụ của Log-Likelihood qua các vòng lặp
plt.figure(figsize=(8, 4.5), dpi=120)
plt.plot(range(1, len(gmm.log_likelihood_history) + 1), gmm.log_likelihood_history, 
         marker='o', color='purple', linewidth=2, markersize=5)
plt.title('Đường cong hội tụ của hàm Log-Likelihood theo vòng lặp EM', fontsize=13, fontweight='bold')
plt.xlabel('Số vòng lặp (Iterations)', fontsize=11)
plt.ylabel('Incomplete Log-Likelihood', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

## 3. Nhận xét chi tiết theo yêu cầu đề bài

### 1. Phân tích Bước E và Bước M:
- **Bước E (Expectation):**
  - Thay vì gán cứng một điểm dữ liệu vào duy nhất một cụm như K-Means, Bước E tính toán **xác suất hậu nghiệm mềm (Soft Assignment)** $\gamma_{ik} = P(z_i = k \mid x_i)$.
  - Một điểm nằm ở vùng biên giữa hai cụm sẽ mang trách nhiệm được chia sẻ (ví dụ $70\%$ thuộc Cụm 1 và $30\%$ thuộc Cụm 2). Điều này phản ánh độ bất định xác thực của dữ liệu thực tế.
- **Bước M (Maximization - Weighted MLE):**
  - Là phiên bản mở rộng của ước lượng hợp lý cực đại có trọng số (Weighted Maximum Likelihood).
  - Tâm cụm $\mu_k$ là trung bình có trọng số của toàn bộ điểm dữ liệu theo trọng số $\gamma_{ik}$.
  - Ma trận hiệp phương sai $\Sigma_k$ ước lượng góc xoay và độ dẹt của elip, cho phép GMM mô hình hóa các cụm có hình dạng elip bất đối xứng và kích thước khác nhau (vượt trội hơn K-Means chỉ mô hình hóa được hình cầu đẳng hướng).

### 2. Sự tiến hóa của các đường đồng mức qua các vòng lặp EM:
- **Vòng lặp 1:** Các tâm cụm được chọn ngẫu nhiên, các ma trận hiệp phương sai ban đầu là hình tròn đẳng hướng bao phủ diện rộng. Các đường đồng mức còn tròn trịa và chưa khớp với phân phối dữ liệu.
- **Vòng lặp 2 - 5:** Các tâm cụm di chuyển nhanh về phía trung tâm của 3 đám mây điểm. Các đường đồng mức bắt đầu biến dạng từ hình tròn thành các hình elip xoay để ôm lấy hình dạng phân tán thực tế của từng cụm.
- **Vòng lặp hội tụ (Vòng 10 - 15):** Các đường đồng mức bám khít hoàn toàn vào 3 đám mây điểm. Các giá trị tâm kỳ vọng $\mu$ và ma trận hiệp phương sai $\Sigma$ tiệm cận chính xác với các tham số gốc sinh dữ liệu (Ground Truth).

### 3. Tính chất hội tụ của Log-Likelihood:
- Đồ thị Log-Likelihood tăng đơn điệu sau mỗi vòng lặp mà không bao giờ bị giảm.
- Về mặt lý thuyết, điều này được đảm bảo bởi thuật toán EM luôn tối ưu hóa chặn dưới **ELBO (Evidence Lower Bound)** của hàm Log-Likelihood, đảm bảo nghiệm hội tụ ổn định về cực trị địa phương (Local Optimum).